In [ ]:
import geopandas as gpd
import rasterio
from rasterio.warp import calculate_default_transform, reproject, Resampling
import numpy as np
import matplotlib.pyplot as plt
from rasterio.features import rasterize
from scipy import ndimage
from osgeo import gdal
from PIL import Image

In [16]:
def calculate_flood_area_local(dem_path, rhein_path, output_path, rise=1.0, layer=None):
    rhein = gpd.read_file(rhein_path, layer = layer)

    with rasterio.open(dem_path) as src:
        dem = src.read(1)
        profile = src.profile.copy()
        nodata = src.nodata
        transform = src.transform
        crs = src.crs

    rhein = rhein.to_crs(crs)

    valid = dem != nodata if nodata is not None else np.ones_like(dem, dtype=bool)

    # Rheinfläche ins Raster brennen
    rhein_mask = rasterize(
        [(geom, 1) for geom in rhein.geometry],
        out_shape=dem.shape,
        transform=transform,
        fill=0,
        dtype="uint8"
    ).astype(bool)

    # Für jedes Pixel: Index des nächstgelegenen Rhein-Pixels
    _, indices = ndimage.distance_transform_edt(
        ~rhein_mask,
        return_indices=True
    )

    nearest_rhein_rows = indices[0]
    nearest_rhein_cols = indices[1]

    # Höhe des nächstgelegenen Rhein-Pixels
    local_rhein_height = dem[nearest_rhein_rows, nearest_rhein_cols]

    # Lokaler Wasserstand = Rheinpixel + 1 m
    local_water_level = local_rhein_height + rise

    # Bathtub-Maske
    potential_flood = (dem <= local_water_level) & valid

    # Nur Flächen behalten, die mit dem Rhein verbunden sind
    labels, _ = ndimage.label(
        potential_flood,
        structure=np.ones((3, 3))
    )

    rhein_labels = np.unique(labels[rhein_mask])
    rhein_labels = rhein_labels[rhein_labels != 0]

    flood = np.isin(labels, rhein_labels)

    # Neues Höhenmodell erstellen
    new_dem = dem.copy().astype("float32")

    # Nur überflutete Pixel auf lokalen Wasserstand setzen
    new_dem[flood] = local_water_level[flood]

    # NoData behandeln
    if nodata is not None:
        new_dem[~valid] = nodata
    else:
        nodata = -9999
        new_dem[~valid] = nodata

    # Wichtig bei VRT als Input
    profile.update({
        "driver": "GTiff",
        "dtype": "float32",
        "count": 1,
        "nodata": nodata,
        "height": new_dem.shape[0],
        "width": new_dem.shape[1],
        "transform": transform,
        "compress": "lzw"
    })

    with rasterio.open(output_path, "w", **profile) as dst:
        dst.write(new_dem, 1)

    print(f"Überflutungsmodell erstellt: {output_path}")

In [42]:
level = 0 # Überflutungslevel anpassen

calculate_flood_area_local(
    dem_path="../Data/Hoehenmodell/Hoehenmodell.vrt",
    rhein_path="../Data/Rhein/rhein.gpkg",
    output_path=f"../Data/Hoehenmodell/Berechnung/rhein_plus{level}m.tif",
    rise = level,
    layer = "Rhein_flaeche_abs",
)

Überflutungsmodell erstellt: ../Data/Hoehenmodell/Berechnung/rhein_plus0m.tif


In [43]:
# Dateien
vrt_pfad = "../Data/Hoehenmodell/Hoehenmodell.vrt"
neu_pfad = f"../Data/Hoehenmodell/Berechnung/rhein_plus{level}m.tif" #Parameter anpassen für die anderen Höhen
ausgabe_pfad = f"../Data/Hoehenmodell/Berechnung/veraenderte_hoehen_{level}m.tif"

# Original DEM laden
with rasterio.open(vrt_pfad) as src_alt:
    alt = src_alt.read(1)
    meta = src_alt.meta.copy()
    original_nodata = src_alt.nodata

# Neues DEM laden
with rasterio.open(neu_pfad) as src_neu:
    neu = src_neu.read(1)

# Falls kein NoData definiert ist
nodata = original_nodata if original_nodata is not None else -9999

# Leeres Raster erstellen
ergebnis = np.full(
    alt.shape,
    nodata,
    dtype=neu.dtype
)

# Geänderte Pixel finden
maske = (
    (np.abs(neu - alt) > 0.01) &
    (alt != nodata) &
    (neu != nodata)
)

# Nur neue Werte übernehmen
ergebnis[maske] = neu[maske]

# Meta für TIFF anpassen
meta.update(
    driver="GTiff",
    nodata=nodata,
    dtype=ergebnis.dtype
)

# Neues Raster schreiben
with rasterio.open(
    ausgabe_pfad,
    "w",
    **meta
) as dst:
    dst.write(ergebnis, 1)

print("Fertig:", ausgabe_pfad)

Fertig: ../Data/Hoehenmodell/Berechnung/veraenderte_hoehen_0m.tif


In [46]:
level = 0

src_path = f"../Data/Hoehenmodell/Berechnung/veraenderte_hoehen_{level}m.tif"
dst_path = f"../Data/Hoehenmodell/Berechnung/veraenderte_hoehen_{level}m_4326.tif"

dst_crs = "EPSG:4326"

with rasterio.open(src_path) as src:

    # Neue Transformation berechnen
    transform, width, height = calculate_default_transform(
        src.crs,
        dst_crs,
        src.width,
        src.height,
        *src.bounds
    )

    # Neues Profil erstellen
    kwargs = src.meta.copy()
    kwargs.update({
        "crs": dst_crs,
        "transform": transform,
        "width": width,
        "height": height
    })

    # Neues TIFF schreiben
    with rasterio.open(dst_path, "w", **kwargs) as dst:

        for i in range(1, src.count + 1):

            reproject(
                source=rasterio.band(src, i),
                destination=rasterio.band(dst, i),
                src_transform=src.transform,
                src_crs=src.crs,
                dst_transform=transform,
                dst_crs=dst_crs,
                resampling=Resampling.nearest
            )

In [71]:
level = 20

tiff = f"../Data/Hoehenmodell/Berechnung/veraenderte_hoehen_{level}m_4326.tif"
png = f"../Data/Hoehenmodell/Berechnung/wasserstandplus{level}m.png"

colors = {
    0: (247,251,255),
    1: (222,235,247),
    2: (198,219,239),
    3: (158,202,225),
    4: (107,174,214),
    5: (66,146,198),
    10: (33,113,181),
    15: (8,81,156),
    20: (8,48,107)
}

r, g, b = colors[level]

with rasterio.open(tiff) as src:
    img = src.read(1)
    mask = img > 0

    rgba = np.zeros((img.shape[0], img.shape[1], 4), dtype=np.uint8)
    rgba[..., 0] = r
    rgba[..., 1] = g
    rgba[..., 2] = b
    rgba[..., 3] = np.where(mask, 255, 0)

Image.fromarray(rgba).save(png)